In [ ]:
import os
import sys
from pathlib import Path

import pytorch_lightning as pl
import torch

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

sys.path.append("../src")

from hand_to_tex.datasets import HMEDataLoaderFactory
from hand_to_tex.models import BaselineTransformer
from hand_to_tex.training import HMELightningModule
from hand_to_tex.utils import LatexVocab

pl.seed_everything(42, workers=True)

if torch.cuda.is_available():
    accelerator = "gpu"
    devices = 1
elif torch.backends.mps.is_available():
    accelerator = "mps"
    devices = 1
else:
    accelerator = "cpu"
    devices = 1

print(f"Accelerator: {accelerator}, devices: {devices}")

## 1. Vocabulary and DataLoaders


In [ ]:
vocab = LatexVocab.load(Path("../data/assets/simple.json"))
print(f"Loaded Vocabulary Dimension Limits: {len(vocab._encode)} items")

root_path = Path("../data/simpler")

factory = HMEDataLoaderFactory(
    root=root_path,
    processed=True,
    vocab=vocab,
    batch_size=32,
    num_workers=7,
    pin_memory=False,
)

train_loader = factory.train()
valid_loader = factory.valid()
test_loader = factory.test()

print(f"Train batches: {len(train_loader)}")
print(f"Valid batches: {len(valid_loader)}")
print(f"Test batches: {len(test_loader)}")

## 2. Model Initialization

In [ ]:
model = BaselineTransformer(
    vocab_size=len(vocab._encode),
    pad_idx=vocab.PAD,
    d_model=64,  # Dimensional size of the abstract mathematical thought state
    nhead=2,  # Number of parallel attention heads running logic checks
    num_encoder_layers=2,  # Transformer blocks decoding the geometric physical handwriting
    num_decoder_layers=2,  # Transformer blocks translating geometry to math text sequence
    in_features=10,  # Initial parameters per point (speed, coordinates, etc.)
)

print(
    f"Model Trainable Constraint count: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} parameters"
)

## 3. Lightning Trainer Execution

In [ ]:
num_epochs = 10

lightning_model = HMELightningModule(
    model=model,
    vocab=vocab,
    lr=5e-5,
    label_smoothing=0.1,
)

trainer = pl.Trainer(
    max_epochs=num_epochs,
    accelerator=accelerator,
    devices=devices,
    gradient_clip_val=1.0,
    log_every_n_steps=10,
    enable_checkpointing=False,
)

trainer.fit(
    model=lightning_model,
    train_dataloaders=train_loader,
    val_dataloaders=valid_loader,
)

trainer.validate(model=lightning_model, dataloaders=valid_loader)
trainer.test(model=lightning_model, dataloaders=test_loader)